# ML that can See: Supervised Learning with Images 

Let's load in any libraries we will use in this notebook. 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

#import torch which has many of the functions to build deep learning models and to train them
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

#import torchvision, which was lots of functions for loading and working with image data
import torchvision
import torchvision.transforms as transforms

#this is a nice progress bar representation that will be good to measure progress during training
import tqdm

# Ingredient 1: The Data
## Inspect the Data
**Make sure you've extracted the dataset folder by right-clicking and selecting 'Extract Archive'**. Once you've done this, look at the different folders and how the dataset is structured. Click open some images to see what they look like and get a feel for the data you're going to be working with.


## Loading the Data

This step has 2 key parts:
1. Create default transformations to apply to the data. The below 3 steps are very standard, and should always be used.
    There are a number of transformations we will consider here, these include:
    1. [transforms.ToTensor()](https://pytorch.org/vision/stable/generated/torchvision.transforms.ToTensor.html) -- this converts a PIL image or numpy array to a tensor while scaling the pixel values to the range [0, 1].
    2. [transforms.Resize()](https://pytorch.org/vision/stable/generated/torchvision.transforms.Resize.html) -- this resizes an input image to the specified size (height, width).
    Resize is important as it ensures the dimensions remain compatible throughout the network, allowing proper operations at each layer and maintaining the required dimensions for the final fully connected layers in the network.
    3. [transforms.Normalize()](https://pytorch.org/vision/stable/generated/torchvision.transforms.Normalize.html) -- this standardizes the pixel values of a tensor image by subtracting the mean and dividing by the standard deviation along the input channels.
    
    You can then use [transforms.Compose](https://pytorch.org/vision/main/generated/torchvision.transforms.Compose.html#torchvision.transforms.Compose) to sequentially chain multiple transforms together.

In [ ]:
imagenet_means = (0.485, 0.456, 0.406)
imagenet_stds = (0.229, 0.224, 0.225)


**Why Resize to 224x224?**
Many popular pre-trained models, such as AlexNet, VGG, and ResNet, were trained on the ImageNet dataset, which used images of size 224x224 pixels. We will use a ResNet architecture pre-trained on ImageNet, so will use this value.

**How do we pick the normalization values?**
We can use the actual underlying statistics in our training data, or we can use the values from the ImageNet dataset (millions of images).


2. Load the datasets in with [torchvision.datasets.ImageFolder](https://pytorch.org/vision/stable/generated/torchvision.datasets.ImageFolder.html) -- this loads image datasets from folders, assigning labels automatically based on subdirectories, making it convenient for tasks like image classification.

Use ImageFolder below to load in the train and val dataset, with the transform from above.

## Visualise the data and class distribution

It's important to get a feel for the data by visualising it, and also to understand any underlying characteristics -- e.g. is there any bias you can detect in the data that might limit how it can be used in the future? what is the balance of different classes in the dataset? 

Below, build a histogram function to visualise the distribution of class labels in the training and validation dataset.

## Creating dataloaders

Dataloaders are Pytorch's useful way of handling data. They automatically batch the dataset into the batch size you want, and can also randomly shuffle the data for you if you choose. They nicely handle the dataset for us during training and testing.

1. *torch.utils.data.DataLoader()* You can read the documentation here: https://pytorch.org/docs/stable/data.html#torch.utils.data.DataLoader
    1. First argument is the dataset.
    2. Optional argument *batch_size* is the batch size to test the model with. How many data points will the model be tested on in parallel?
    3. Optional argument *shuffle* controls whether data is randomly shuffled before taking from the dataset.
    4. Optional argument *num_workers* is how many subprocesses are used to load data from the dataset -- it can make loading the data faster.

Create a dataloader for the train and validation dataset. Use a batch size of 16.

# Ingredient 2: The Model

## Initialise the model
We will use a pretrained ResNet18, that has been trained on ImageNet. This is very easy to do in PyTorch -- ```torchvision.models.resnet18``` loads the architecture, and using ```weights=ResNet18_Weights.DEFAULT``` loads the trained parameters for the model after it was trained on ImageNet.

You can see all the models built into torchvision [here](https://pytorch.org/vision/stable/models.html#classification).


## Adapt the model architecture for transfer learning

In the printed model definition above, the final Linear layer is taking an input of size 512 (in_features) and using 1000 neurons (out_features) to create 1000 class scores. The model was created this way because it was trained to perform image classification on ImageNet, which has 1000 classes.

We have a variable ```num_classes``` that is storing how many classes our new dataset has, and how many class scores we want to generate. 
Let's re-assign the final layer by creating a new linear layer using [nn.Linear](https://pytorch.org/docs/stable/generated/torch.nn.Linear.html). The number of input features will not change, but the number of neurons or out_features should. When you print(model.fc), you should see a Linear layer with 512 in_features and 13 out_features.

In [ ]:
in_features = model.fc.in_features
model.fc = nn.Linear(model.fc.in_features, num_classes)

print(model.fc)

Now that our model is adapted for transfer learning, we should also load the model onto our GPU if it is available -- this will massively speed up training.

In [ ]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu') #this line checks if we have a GPU available
model = model.to(device)

# Training Time: Transfer Learning with our new dataset



## Transfer Learning with fine-tuning
Now that we've initialised our model, adapted it's architecture for our new training dataset, and initialised the other elements of training (stochastic gradient descent optimizer and cross-entropy loss), we can start to train our model.

We're first going to use a fine-tuning approach, where the model's parameters are adjusted slightly to adapt its learned features to the specific nuances of the new task or domain. We're going to adjust the parameters in every layer of the network (i.e. do not freeze any layers).

In the Week 3 lecture and tutorial, we learnt that we train our model by:
1. Define a loss function (or cost function or objective).
2. Initialise the SGD optimizer.
3. Until satisfied (e.g. loss converged/stops changing):
    1. Perform a forward pass to find a prediction.
    2. Calculate the loss
    3. Perform a backward pass to calculate loss gradients with respect to the parameters.
    4. Update the parameters based on the loss gradients, using our SGD optimize.


We should also monitor performance on the validation dataset as we go to identify overfitting.

Let's build this training code up again below.


In [ ]:
torch.manual_seed(0)


## Re-loading our training dataset with data augmentations

There are a couple of things that can help with overfitting -- one would be adding some data transformations. 

Go through the data transformation functions below, and choose some additional transforms to apply. Remember to tailor the transformations to the characteristics the dataset. We also only apply data augmentation to the training set, while the validation and test sets will use the standard transform that does not include data augmentation.

**Your turn: Pick additional data transforms and add them to the train_transforms list!**

1. (already implemented below) [transforms.RandomResizedCrop](https://pytorch.org/vision/stable/generated/torchvision.transforms.RandomResizedCrop.html) -- this function randomly grabs a portion of the image (crops) and then resizes to the desired image size. By default, the crop can be anywhere between 8% to 100% of the image original area -- this is a little strict, I'm going to choose between 50% and 100% of the image area.
2. [transforms.RandomHorizontalFlip](https://pytorch.org/vision/stable/generated/torchvision.transforms.RandomHorizontalFlip.html) -- randomly flips an image horizontally. Useful for tasks where horizontal orientation doesn't change the meaning.
3. [transforms.RandomVerticalFlip](https://pytorch.org/vision/stable/generated/torchvision.transforms.RandomVerticalFlip.html) -- randomly flips an image vertically. Useful for tasks where vertical orientation doesn't change the meaning.
4. [transforms.RandomRotation](https://pytorch.org/vision/stable/generated/torchvision.transforms.RandomRotation.html) -- randomly rotates an image by a specified angle. Can simulate variations in viewpoint.
5. [transforms.ColorJitter](https://pytorch.org/vision/stable/generated/torchvision.transforms.ColorJitter.html) -- randomly changes brightness, contrast, saturation, and hue of an image. Helps the model to be robust to different lighting conditions.


Once we've done this, there's a few way to use this transform -- we can add it into the training loop (what we will do), or you can make a custom dataset that automatically applies this transformation for the training subset (see here: https://discuss.pytorch.org/t/transforms-on-subset/166836)

**What's going on here?**

Each of the 'Random' transformations will be sequentially applied to an input image, with different transformations of different severities - the severity of the transformation is the random component. When you chain together multiple different types of 'Random' transformations, we can end up with a huge variation of different images from our training dataset.

In [ ]:
### Add more transforms in here
train_transform = transforms.Compose(
    [....
    ])


#visualise the train dataset with these transforms
data = next(iter(trainloader))
fig, ax = plt.subplots(1, 5)
for idx in range(5):
    im = data[0][idx]
    lbl = data[1][idx]
    im = train_transform(im)
    train_image = (im.numpy())/2 + 0.5
    label = class_labels[lbl]
    train_image = np.moveaxis(train_image, 0, 2)
    ax[idx].imshow(train_image)
    ax[idx].set_axis_off()
    ax[idx].set_title(label.split('-')[-1])
plt.tight_layout()
plt.show()


## Train with Data Augmentation

Copy the training code from the cell above and train with the train_transform.

Make sure to add it into the training loop before you pass the training inputs to the model.

## Food for thought
1. Did your data augmentations improve generalisation? If not -- this can happen! There is always some experimentation with this process, and often your first attempt doesn't work.
2. Can you justify why you picked the data augmentation transformations that you used? Why would these transformations provide better generalisation for this dataset? This will be an important thing to think about for Project 1.

## Accounting for class imbalance

When we're training our model, every image in the batch is treated equally important for training.

So what happens when we have some classes with very little data, and others with **loads** of data? Potentially our model will overly focus on performing well on the class with lots of data, and neglect the class with less data.

We can account for this using Pytorch's [WeightedRandomSampler](https://pytorch.org/docs/stable/data.html#torch.utils.data.WeightedRandomSampler), which can be used in the DataLoader class.

To do this, we need to choose weights for the sampler (these dictate how often certain samples are used in a batch) based on our class imbalance, create the WeightedRandomSampler, and reload our Dataloader. Let's do this below, and then try re-training our model.

In [ ]:
train_labels = train_dataset.targets

lbls, counts = np.unique(train_labels, return_counts = True)

weighting = torch.DoubleTensor([1/x for x in counts])
sample_weights = weighting[train_dataset.targets]

sampler = torch.utils.data.WeightedRandomSampler(sample_weights, len(train_dataset))


trainloader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size,
                                          sampler = sampler)


## Transfer Learning with freezing layers

Freezing layers in a PyTorch model involves setting the ```requires_grad``` (i.e. requires gradient) attribute of the parameters in those layers to ```False```. This prevents the parameters from being updated during training by the optimizer. You can selectively freeze layers and train only the desired layers, such as the last fully connected layer. 

Below, I'm first re-initialising a fresh instance of ResNet18. I'm then going through every layer in the model and setting the ```requires_grad``` attribute to False. Then, I change only the last fully-connected layer ```requires_grad``` attribute to True.

If you want to look at model parameter names, you can always ```print(model)```. 

In [ ]:
model = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.IMAGENET1K_V1)
in_features = model.fc.in_features
model.fc = nn.Linear(model.fc.in_features, num_classes)

model.to(device)


for param in model.parameters():
    param.requires_grad = False

# Unfreeze the parameters of the last fully connected layer
for param in model.fc.parameters():
    param.requires_grad = True

Grab your training code from earlier and re-train the model, now with a frozen backbone. Does the performance get better or worse?

You may need to change how long you train for and your learning rate.